### 졸음감지 전이학습 이미지 데이터 전처리 

In [115]:
import os
import cv2
import copy 
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import torch 
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [36]:
# check length of datasets 
dset1_close = os.listdir('./dataset_1/train/Closed_Eyes')
dset1_open = os.listdir('./dataset_1/train/Open_Eyes')
dset2_tr_close = os.listdir('./dataset_2/TrainingSet/TrainingSet/Closed')
dset2_tr_open = os.listdir('./dataset_2/TrainingSet/TrainingSet/Opened')
dset2_ts_close = os.listdir('./dataset_2/TestSet/TestSet/Closed')
dset2_ts_open = os.listdir('./dataset_2/TestSet/TestSet/Opened')
dset3_tr_close = os.listdir('./dataset_3/dataset_new/train/Close')
dset3_tr_open = os.listdir('./dataset_3/dataset_new/train/Open')
dset3_ts_close = os.listdir('./dataset_3/dataset_new/test/Close')
dset3_ts_open = os.listdir('./dataset_3/dataset_new/test/Open')

print('dataset1 closed:', len(dset1_close))
print('dataset1 open:', len(dset1_open))
print('dataset2 train closed:',len(dset2_tr_close))
print('dataset2 train open:', len(dset2_tr_open))
print('dataset2 test closed:', len(dset2_ts_close))
print('dataset2 test open:', len(dset2_ts_open))
print('dataset3 train close:', len(dset3_tr_close))
print('dataset3 train open:', len(dset3_tr_open))
print('dataset3 test close:', len(dset3_ts_close))
print('dataset3 test open:', len(dset3_ts_open))

FileNotFoundError: [Errno 2] No such file or directory: './dataset_1/train/Closed_Eyes'

In [116]:
train_dir = './train/'
test_dir = './test/'
dset1_dir = './dataset_1/train/'

In [117]:
def list_image_files(data_dir, sub_dir, tr_ts=None):
    image_format = ['jpeg', 'jpg', 'png']
    image_files = []
    image_dir = os.path.join(data_dir, sub_dir)
    for i, file_path in enumerate(os.listdir(image_dir)):
        if tr_ts == None:
            if file_path.split('.')[-1] in image_format:
                image_files.append(os.path.join(sub_dir, file_path))
        elif tr_ts == 'test':
            if i%4 == 0:
                if file_path.split('.')[-1] in image_format:
                    image_files.append(os.path.join(sub_dir, file_path))        
        elif tr_ts == 'train':
            if i%4 != 0:
                if file_path.split('.')[-1] in image_format:
                    image_files.append(os.path.join(sub_dir, file_path))
    return image_files 

In [118]:
# len(list_image_files(dset1_dir, 'open', tr_ts= 'test'))

500

In [119]:
def get_RGB_image(data_dir, file_name):
    image_file = os.path.join(data_dir, file_name)
    image = cv2.imread(image_file)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image

In [120]:
transformer = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224,224)),
    transforms.Normalize(mean=[0.5,0.5,0.5], std = [0.5,0.5,0.5])
])

In [121]:
class Eye_dataset(Dataset):
    def __init__(self, data_dir, dset1_dir, tr_ts, transform=None): 
        self.data_dir = data_dir
        self.dset1_dir = dset1_dir 
        self.classes = ['close', 'open']
        self.close = list_image_files(dset1_dir, 'close', tr_ts) + list_image_files(data_dir, 'close') 
        self.open = list_image_files(dset1_dir, 'open', tr_ts) + list_image_files(data_dir, 'open')  
        self.files_path = self.close + self.open 
        self.transforms = transform 
        
    def __len__(self):
        return len(self.files_path)

    def __getitem__(self, index):
        if self.files_path[index].split('.')[-1] == 'png':
           image_file = os.path.join(self.dset1_dir, self.files_path[index])
        else:
           image_file = os.path.join(self.data_dir, self.files_path[index])
        image = cv2.imread(image_file)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        target = self.classes.index(self.files_path[index].split(os.sep)[-2])
        
        if self.transforms:
            image = self.transforms(image)
            target = torch.Tensor([target]).long()
        
        return {'image':image, 'target':target}

In [122]:
dset1 = Eye_dataset(train_dir,  dset1_dir, 'train',transformer )
#print(len(dset3))
#print(dset3[5000]['image'].shape)

8466
torch.Size([3, 224, 224])


In [123]:
dset2 =  Eye_dataset(test_dir,  dset1_dir, 'test',transformer )
#print(len(dset4))
#print(dset4[20]['image'].shape)

2922
torch.Size([3, 224, 224])


In [124]:
def build_dataloader(train_dir, val_dir, dset1_dir, tr, ts ): # tr: 'traing' , ts: 'test'
    dataloaders = {}
    
    train_dset = Eye_dataset(train_dir, dset1_dir, tr, transformer)
    
    dataloaders['train'] = DataLoader(train_dset, batch_size=4, shuffle =True, drop_last = True)
    
    val_dset = Eye_dataset(val_dir, dset1_dir, ts, transformer)
    
    dataloaders['val'] = DataLoader(val_dset, batch_size=4, shuffle =False, drop_last = False)
    
    return dataloaders

In [125]:
train_dir = './train/'
test_dir = './test/'
dset1_dir = './dataset_1/train/'
val_dir = test_dir

In [126]:
dataloaders = build_dataloader(train_dir, val_dir, dset1_dir, 'train', 'tset')

In [131]:
for i , n in enumerate(dataloaders['train']):
    print(i, n)
    if i == 0:
        break
        
print(n['image'].shape)  # torch.Size([4, 3, 224, 224])

0 {'image': tensor([[[[-0.4745, -0.4768, -0.4861,  ..., -0.0744, -0.0682, -0.0667],
          [-0.4745, -0.4768, -0.4865,  ..., -0.0740, -0.0681, -0.0667],
          [-0.4745, -0.4771, -0.4880,  ..., -0.0725, -0.0678, -0.0667],
          ...,
          [-0.3450, -0.3462, -0.3509,  ..., -0.1625, -0.1485, -0.1451],
          [-0.3419, -0.3434, -0.3493,  ..., -0.1609, -0.1482, -0.1451],
          [-0.3412, -0.3427, -0.3489,  ..., -0.1606, -0.1481, -0.1451]],

         [[-0.4745, -0.4768, -0.4861,  ..., -0.0744, -0.0682, -0.0667],
          [-0.4745, -0.4768, -0.4865,  ..., -0.0740, -0.0681, -0.0667],
          [-0.4745, -0.4771, -0.4880,  ..., -0.0725, -0.0678, -0.0667],
          ...,
          [-0.3450, -0.3462, -0.3509,  ..., -0.1625, -0.1485, -0.1451],
          [-0.3419, -0.3434, -0.3493,  ..., -0.1609, -0.1482, -0.1451],
          [-0.3412, -0.3427, -0.3489,  ..., -0.1606, -0.1481, -0.1451]],

         [[-0.4745, -0.4768, -0.4861,  ..., -0.0744, -0.0682, -0.0667],
          [-0.4745

In [130]:
for i , n in enumerate(dataloaders['val']):
    print(i, n)
    if i == 0:
        break
print(n['image'].shape)

0 {'image': tensor([[[[ 0.6425,  0.6252,  0.6051,  ..., -0.1570, -0.2035, -0.2453],
          [ 0.6395,  0.6488,  0.6353,  ..., -0.1896, -0.2154, -0.2416],
          [ 0.6504,  0.6629,  0.6473,  ..., -0.1711, -0.1891, -0.2221],
          ...,
          [ 0.1356,  0.1315,  0.1354,  ..., -0.3028, -0.6276, -0.8200],
          [ 0.1281,  0.1404,  0.1474,  ..., -0.3858, -0.6557, -0.8062],
          [ 0.1301,  0.1551,  0.1550,  ..., -0.4689, -0.7054, -0.8227]],

         [[ 0.4856,  0.4684,  0.4482,  ..., -0.2590, -0.2939, -0.3238],
          [ 0.4827,  0.4920,  0.4784,  ..., -0.2916, -0.3058, -0.3200],
          [ 0.4936,  0.5060,  0.4904,  ..., -0.2731, -0.2795, -0.3005],
          ...,
          [-0.0840, -0.0881, -0.0842,  ..., -0.3628, -0.6638, -0.8442],
          [-0.0915, -0.0792, -0.0722,  ..., -0.4392, -0.6832, -0.8160],
          [-0.0895, -0.0645, -0.0646,  ..., -0.5057, -0.7212, -0.8306]],

         [[ 0.3994,  0.3821,  0.3619,  ..., -0.3217, -0.3605, -0.3944],
          [ 0.3964